# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K-0-A/My-Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

####1. what one row means for your lane: One row = one page-day. Specifically, a single content_hash_id for a specific client_hash_id on a specific report_date.
####2. which table(s) you'll use:  fact_content_daily_performance from the Hugging Face warehouse release.
####3. which time window: I will use a mid-panel month, month=2026-03. I am deliberately avoiding the _sample table because it represents the final month (June 2026), which is the natural outcome window for a past→future label. Iterating there would cause data leakage.
####4.what you'd predict or rank (label or proxy):  Because this is daily data, I will define a past→future proxy: whether a page's gsc_clicks drop by more than 20% in the 30 days after the decision moment compared to the 30 days before.
####5.one thing you deliberately exclude:  I exclude any metrics from the future outcome window (to prevent target leakage), and I exclude rows where ga4_data_available IS FALSE when computing engagement features, because those zeros are fake.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

####1. Context (never features): report_date (time indexing), client_hash_id (grouped splits), content_hash_id (joining/grouping).
####2. Features (knowable at decision moment): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions (filtered by availability), scroll_events.
####3.Label / Proxy: My derived future drop in gsc_clicks.
####4.Excluded:
Any future window data (leakage).

Raw GA4 metrics where ga4_data_available IS FALSE (fake zeros).

gsc_sum_position (redundant; we use gsc_avg_position).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

####1. Build a small feature frame for your lane from that same month, and give every feature one line
#####A. gsc_avg_position_prev_30d — Knowable at the decision moment because it is the average position strictly calculated from the 30 days prior to the prediction date.
#####B. content_age_days — Knowable because it is calculated as report_date minus the page's first-ever report date, which is historical fact.
#####C. ga4_sessions_prev_30d — Knowable because it aggregates sessions from the past window (filtered by ga4_data_available IS TRUE).
#####D. gsc_clicks_prev_30d — Knowable because it counts clicks from the past window only.
#####E. gsc_impressions_prev_30d — Knowable because it aggregates search impressions from the past window only.


In [8]:
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

# Never paste your token in the cell! Use Colab Secrets.
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Point to the mid-panel month
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

In [9]:
# Grain check: Does one row = one page-day?
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate rows found: {len(grain_check)}")
if len(grain_check) == 0:
    print("Grain holds: one row = one page-day.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found: 0
Grain holds: one row = one page-day.


In [10]:
# Counts and window
counts = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_hash_id) as unique_clients,
        COUNT(DISTINCT content_hash_id) as unique_pages,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date
    FROM {REL}
""").df()

print(counts.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_clients  unique_pages   min_date   max_date
    9841378              55        331437 2026-03-01 2026-03-31


In [11]:
# Availability check: GA4 rows with IS TRUE
availability = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows
    FROM {REL}
""").df()

print(availability.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  gsc_available_rows
    9841378            413966.0           3611061.0


In [13]:
# Build the 5-feature frame for a single decision date (e.g., March 31, 2026)
feature_frame = con.sql(f"""
    WITH past_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_avg_position) as gsc_avg_position_prev_30d,
            SUM(gsc_impressions) as gsc_impressions_prev_30d,
            SUM(gsc_clicks) as gsc_clicks_prev_30d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) as ga4_sessions_prev_30d
        FROM {REL}
        WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-30'
        GROUP BY client_hash_id, content_hash_id
    ),
    age_calc AS (
        SELECT
            client_hash_id,
            content_hash_id,
            MIN(report_date) as first_seen_date
        FROM {REL}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.gsc_avg_position_prev_30d,
        p.gsc_impressions_prev_30d,
        p.gsc_clicks_prev_30d,
        p.ga4_sessions_prev_30d,
        DATE '2026-03-31' - a.first_seen_date AS content_age_days
    FROM past_window p
    LEFT JOIN age_calc a USING (client_hash_id, content_hash_id)
    LIMIT 1000
""").df()

print(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            client_hash_id           content_hash_id  \
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  client_73cda7b4e4f265ea  content_905aa32a0230694e   
3  client_73cda7b4e4f265ea  content_05434271b257bb68   
4  client_73cda7b4e4f265ea  content_d056587ff7faca0c   

   gsc_avg_position_prev_30d  gsc_impressions_prev_30d  gsc_clicks_prev_30d  \
0                   4.388327                    1105.0                  2.0   
1                   2.463333                      54.0                  0.0   
2                   6.704951                     148.0                  0.0   
3                   6.374605                    1369.0                  6.0   
4                   4.396002                    2682.0                 16.0   

   ga4_sessions_prev_30d  content_age_days  
0                    0.0                30  
1                    0.0                30  
2                    4.0                30  
3       

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

####1. Add ONE label-derived column on purpose:
####Part 1: the Data limit
Limitation: A massive 96% of the rows in this month have ga4_data_available = FALSE (only 413k out of 9.8M rows have real GA4 data). If I do not explicitly filter on this boolean flag, any engagement metric (like sessions or pageviews) will be falsely dragged toward zero, making healthy pages look dead.

####Part 2: The Trap (Leakage)
The Trap: To prove the danger of leakage, I will intentionally add a label-derived column (future_clicks_drop). I will train a quick Decision Tree using this leaky feature. The model will achieve near-perfect accuracy because it is literally reading the answer. Then, I will delete it and show the honest, lower score.

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Create a mock label and a LEAKY feature (the future)
# For demonstration, we pretend the label is "clicks dropped in the next 30 days"
# And the leaky feature is the future clicks.
leak_test = con.sql(f"""
    WITH past_clicks AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as past_clicks
        FROM {REL} WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-30'
        GROUP BY 1, 2
    ),
    future_clicks AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as future_clicks
        FROM {REL} WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT p.past_clicks, f.future_clicks,
           CASE WHEN f.future_clicks < p.past_clicks * 0.8 THEN 1 ELSE 0 END as is_declining
    FROM past_clicks p
    JOIN future_clicks f USING (client_hash_id, content_hash_id)
    WHERE p.past_clicks > 0
    LIMIT 10000
""").df()

# Honest model (using only past data)
X_honest = leak_test[['past_clicks']]
y = leak_test['is_declining']
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_model.fit(X_train, y_train)
honest_acc = accuracy_score(y_test, honest_model.predict(X_test))
print(f"Honest Model (Past Clicks only) Accuracy: {honest_acc:.3f}")

# Leaky model (using the future answer!)
X_leaky = leak_test[['past_clicks', 'future_clicks']]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_model.fit(X_train_l, y_train_l)
leaky_acc = accuracy_score(y_test_l, leaky_model.predict(X_test_l))
print(f"Leaky Model (Past + Future Clicks) Accuracy: {leaky_acc:.3f}  <-- Leakage!")

# The fix: Delete the leaky feature and keep the honest number.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Model (Past Clicks only) Accuracy: 1.000
Leaky Model (Past + Future Clicks) Accuracy: 1.000  <-- Leakage!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.